# انتقال پلیر آوا به گوگل‌درایو

این صفحه نسخهٔ دوم **0.2.0** را از GitHub مستقیماً به درایو خودتان کپی می‌کند؛ فایل روی گوشی دانلود نمی‌شود.

1. از منوی **Runtime → Run all** استفاده کنید، یا دکمهٔ ▶ کنار کادر پایین را بزنید.
2. در پنجرهٔ رسمی گوگل، حساب مربوط به «آوا مدیا» را انتخاب و اتصال درایو را تأیید کنید.
3. منتظر پیام «فایل با موفقیت در درایو ذخیره شد» بمانید.

پوشهٔ پیش‌فرض **My Drive / آوا مدیا** است؛ می‌توانید `folder_path` را پیش از اجرا تغییر دهید. اگر پوشه نباشد ساخته می‌شود. پوشه‌های دیگر جست‌وجو نمی‌شوند. فایل هم‌نام متفاوت بازنویسی نمی‌شود و فایل اصلی گیت‌هاب دست‌نخورده می‌ماند. این کد نصاب را اجرا نمی‌کند.

حجم حدود ۱۸۶ مگابایت است. صحت فایل پیش و پس از کپی با SHA256 نسخهٔ منتشرشده بررسی می‌شود. اطلاعات ورود گوگل به چت یا این پروژه فرستاده نمی‌شود؛ مجوز اتصال از رابط خود گوگل دریافت می‌شود. تا زمان اجرای موفق این صفحه، آپلود به درایو انجام نشده است.


In [ ]:
# @title انتقال نسخهٔ دوم پلیر آوا به گوگل‌درایو
# مسیر پوشه نسبت به My Drive؛ در صورت نیاز مسیر دلخواه را وارد کنید.
folder_path = "آوا مدیا" # @param {type:"string"}

"""Copy the published AVA installer to a mounted Drive, verifying its checksum.

This module never executes the installer and never handles Google credentials.
Google Colab mounts Drive through Google's own authorization screen.
"""
import hashlib
import shutil
import tempfile
from pathlib import Path
from urllib.request import Request, urlopen

VERSION = '0.2.0'
ASSET = f'AVA-Player-Setup-{VERSION}-x64.exe'
URL = f'https://github.com/hassandana1371-hub/ava-player/releases/download/v{VERSION}/{ASSET}'
SHA256 = '83e4275c069ae8d0b6d503dc1223fad9fde7c940720ed91ed654d4d03780aa4b'
SIZE = 194813807


def digest(file):
    with Path(file).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def transfer(url, destination, expected_hash, expected_size, *, opener=urlopen, progress=print):
    """Return True for a new verified copy, False for an identical existing file.

    Conflicting existing files are preserved. A download must pass both length
    and SHA256 checks before anything is written to the destination directory.
    """
    destination = Path(destination)
    if destination.exists():
        if destination.is_file() and destination.stat().st_size == expected_size and digest(destination) == expected_hash:
            return False
        raise FileExistsError('یک فایل متفاوت با همین نام وجود دارد؛ مسیر دیگری انتخاب کنید.')
    with tempfile.TemporaryDirectory(prefix='ava-drive-') as directory:
        downloaded = Path(directory) / 'installer.download'
        hasher = hashlib.sha256()
        received = 0
        last_report = 0
        request = Request(url, headers={'User-Agent': 'AVA-Player-Drive-Transfer/0.2'})
        with opener(request, timeout=60) as response, downloaded.open('wb') as output:
            while block := response.read(1024 * 1024):
                received += len(block)
                if received > expected_size:
                    raise ValueError('حجم فایل دریافت‌شده با نسخهٔ تأییدشده مطابقت ندارد.')
                output.write(block)
                hasher.update(block)
                if received - last_report >= 32 * 1024 * 1024:
                    progress(f'دریافت: {received // (1024 * 1024)} مگابایت')
                    last_report = received
        if received != expected_size or hasher.hexdigest() != expected_hash:
            raise ValueError('بررسی صحت فایل موفق نبود؛ فایل به درایو منتقل نشد.')
        destination.parent.mkdir(parents=True, exist_ok=True)
        created = False
        try:
            # Exclusive creation also protects against a concurrent upload.
            with destination.open('xb') as output, downloaded.open('rb') as incoming:
                created = True
                shutil.copyfileobj(incoming, output)
            if digest(destination) != expected_hash:
                raise ValueError('بررسی فایل کپی‌شده در درایو موفق نبود.')
        except BaseException:
            if created:
                destination.unlink(missing_ok=True)
            raise
    return True


from google.colab import drive

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive').resolve()
folder = (root / folder_path).resolve()
if not folder.is_relative_to(root):
    raise ValueError('مسیر پوشه باید داخل My Drive باشد.')
print('انتقال مستقیم از گیت‌هاب به درایو؛ نیازی به دانلود روی گوشی نیست.')
copied = transfer(URL, folder / ASSET, SHA256, SIZE)
print('فایل با موفقیت در درایو ذخیره و صحت آن بررسی شد.' if copied else 'همین فایل از قبل در این پوشه موجود است؛ کپی مجدد لازم نبود.')
print('مسیر در درایو:', str((folder / ASSET).relative_to(root)))
